In [13]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [22]:
words = open('../names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [23]:
len(words)

32033

In [24]:
# build the vocabulary of characters
chars = sorted(list(set(''.join(words))))
stoi = {s: i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}
vocab_size = len(itos)
print(itos)
print(vocab_size)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
27


In [25]:
# build dataset
block_size = 3

def build_dataset(words):
    X, Y = [], []

    for w in words:
        context = [0] * block_size 
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]

    X = torch.tensor(X)
    Y = torch.tensor(Y)

    print(X.shape, Y.shape)
    return X, Y 

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))


Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte =  build_dataset(words[n2:])

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [26]:
n_emd = 10
n_hidden = 200

g = torch.Generator().manual_seed(45)

C = torch.randn((vocab_size, n_emd), generator=g)
W1 = torch.randn((n_emd * block_size, n_hidden), generator=g)
b1 = torch.randn(n_hidden, generator=g)
W2 = torch.randn((n_hidden, vocab_size), generator=g)
b2 = torch.randn(vocab_size, generator=g)

parameters = [C, W1, b1, W2, b2]
print(sum(p.nelement() for p in parameters))
for p in parameters:
    p.requires_grad = True
    

11897


In [27]:
max_steps = 200000
batch_size = 32
lossi = []

for i in range(max_steps):
    
    # mini batch
    ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
    Xb, Yb = Xtr[ix], Ytr[ix] # batch of X, y 

    # forward pass
    emb = C[Xb]
    embcat = emb.view(emb.shape[0], -1)
    hpreact = embcat @ W1 + b1
    h = torch.tanh(hpreact)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Yb)


    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()


    # update
    lr = 0.1 if i < 100000 else 0.01
    for p in parameters:
        p.data += -lr * p.grad
    
    # track stats 
    if i % 10000 == 0:
        print(f'{i:7d} / {max_steps:7d}: {loss.item():.4f}')
    lossi.append(loss.log10().item())

      0 /  200000: 26.0468
  10000 /  200000: 2.6418
  20000 /  200000: 2.6236
  30000 /  200000: 2.3134
  40000 /  200000: 2.3964
  50000 /  200000: 2.4406
  60000 /  200000: 2.1491
  70000 /  200000: 2.0906
  80000 /  200000: 2.3629
  90000 /  200000: 2.1968
 100000 /  200000: 2.3960
 110000 /  200000: 1.8520
 120000 /  200000: 2.0035
 130000 /  200000: 2.0937
 140000 /  200000: 2.2909
 150000 /  200000: 2.3351
 160000 /  200000: 1.8970
 170000 /  200000: 2.0129
 180000 /  200000: 2.1132
 190000 /  200000: 2.3931


In [28]:
@torch.no_grad()
def split_loss(split):
    x, y =  {
        'train':(Xtr, Ytr),
        'val':(Xdev, Ydev),
        'test':(Xte, Yte),
    }[split]

    emb = C[x]
    embcat = emb.view(emb.shape[0], -1)
    h = torch.tanh(embcat @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, y)
    print(split, loss.item())
    
split_loss('train')
split_loss('val')

train 2.114931344985962
val 2.162468194961548


In [29]:
g = torch.Generator().manual_seed(45 + 10)

for _ in range(20):
    out = []
    context = [0] * block_size

    while True:
        # forward pass for neural net
        emb = C[torch.tensor([context])]
        h = torch.tanh(emb.view(1, -1) @ W1 + b1)
        logits = h @ W2 + b2
        probs = F.softmax(logits, dim=1)

        ix = torch.multinomial(probs, num_samples=1, generator=g).item()
        context = context[1:] + [ix]
        out.append(ix)

        if ix == 0:
            break
    
    print(''.join(itos[i] for i in out))
    


milosophyne.
keemulee.
georaodar.
ai.
alie.
emmalletha.
kiyannesie.
amor.
eva.
chrinchioushfrri.
than.
tabenotnie.
madalyx.
kedy.
eux.
aah.
tukanson.
jermielli.
gekersnha.
sera.
